# Exp3: CGM + time_since_meal + image (resnet18, gated)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+resnet18 (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'resnet18'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_resnet18'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_resnet18_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_resnet18_val_stability_summary.json'

In [3]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print()
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print()
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]


number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 512), (256, 5)]


In [4]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=512, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_resnet' /f'gated_cgm_time_resnet18_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_resnet18_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Validation results — CGM+time+resnet18 (gated) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,260,757


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.62 | MAE 8.55 | gRMSE 13.71 | Zone A 94.72% | A+B 99.54%
 30 min | RMSE 18.28 | MAE 12.27 | gRMSE 20.62 | Zone A 88.65% | A+B 99.27%
 60 min | RMSE 25.80 | MAE 17.14 | gRMSE 30.46 | Zone A 80.00% | A+B 98.74%
 90 min | RMSE 30.36 | MAE 20.20 | gRMSE 36.87 | Zone A 75.05% | A+B 98.20%
120 min | RMSE 33.11 | MAE 22.53 | gRMSE 40.56 | Zone A 70.98% | A+B 97.81%

--------------------------------------------------------------------------------
Validation results — CGM+time+resnet18 (gated) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,260,757


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.72 | MAE 8.04 | gRMSE 12.60 | Zone A 95.93% | A+B 99.72%
 30 min | RMSE 17.86 | MAE 12.01 | gRMSE 19.88 | Zone A 89.31% | A+B 99.33%
 60 min | RMSE 25.76 | MAE 17.00 | gRMSE 30.25 | Zone A 80.38% | A+B 98.71%
 90 min | RMSE 30.66 | MAE 20.49 | gRMSE 37.01 | Zone A 74.60% | A+B 98.15%
120 min | RMSE 33.47 | MAE 22.79 | gRMSE 40.84 | Zone A 70.63% | A+B 97.79%

--------------------------------------------------------------------------------
Validation results — CGM+time+resnet18 (gated) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,260,757


...

 15 min | RMSE 11.56 | MAE 7.88 | gRMSE 12.57 | Zone A 95.83% | A+B 99.53%
 30 min | RMSE 17.73 | MAE 11.90 | gRMSE 19.98 | Zone A 89.10% | A+B 99.28%
 60 min | RMSE 25.56 | MAE 16.84 | gRMSE 30.13 | Zone A 80.89% | A+B 98.71%
 90 min | RMSE 30.42 | MAE 20.12 | gRMSE 36.70 | Zone A 75.50% | A+B 98.14%
120 min | RMSE 33.40 | MAE 22.65 | gRMSE 40.67 | Zone A 71.01% | A+B 97.81%

--------------------------------------------------------------------------------
Validation results — CGM+time+resnet18 (gated) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,260,757


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.13 | MAE 7.55 | gRMSE 12.30 | Zone A 96.53% | A+B 99.64%
 30 min | RMSE 17.85 | MAE 11.94 | gRMSE 20.24 | Zone A 88.93% | A+B 99.32%
 60 min | RMSE 26.23 | MAE 17.37 | gRMSE 31.04 | Zone A 79.32% | A+B 98.65%
 90 min | RMSE 30.86 | MAE 20.50 | gRMSE 37.32 | Zone A 73.95% | A+B 98.15%
120 min | RMSE 33.21 | MAE 22.48 | gRMSE 40.57 | Zone A 71.07% | A+B 97.84%

--------------------------------------------------------------------------------
Validation results — CGM+time+resnet18 (gated) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,260,757


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.40 | MAE 7.80 | gRMSE 12.60 | Zone A 96.36% | A+B 99.65%
 30 min | RMSE 17.94 | MAE 12.07 | gRMSE 20.51 | Zone A 89.40% | A+B 99.30%
 60 min | RMSE 25.98 | MAE 17.07 | gRMSE 31.07 | Zone A 80.19% | A+B 98.69%
 90 min | RMSE 30.71 | MAE 20.38 | gRMSE 37.33 | Zone A 74.78% | A+B 98.18%
120 min | RMSE 33.50 | MAE 22.67 | gRMSE 41.11 | Zone A 70.71% | A+B 97.80%

--------------------------------------------------------------------------------


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [7]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [8]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== CGM+time+resnet18 (gated) validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.72 +/- 0.51 (2.31) mg/dL   MAE = 8.02 +/- 0.35 (1.56)   gRMSE = 12.76 +/- 0.48 (3.21)   Zone A = 95.85 +/- 0.65 (3.91)%   Zone A+B = 99.61 +/- 0.07 (0.73)%
 30-min:  RMSE = 17.91 +/- 0.20 (3.87) mg/dL   MAE = 12.03 +/- 0.13 (2.53)   gRMSE = 20.22 +/- 0.24 (5.84)   Zone A = 89.18 +/- 0.46 (5.37)%   Zone A+B = 99.30 +/- 0.02 (1.06)%
 60-min:  RMSE = 25.82 +/- 0.18 (7.54) mg/dL   MAE = 17.02 +/- 0.18 (5.00)   gRMSE = 30.60 +/- 0.34 (11.63)   Zone A = 80.44 +/- 0.60 (5.80)%   Zone A+B = 98.70 +/- 0.03 (1.40)%
 90-min:  RMSE = 30.62 +/- 0.15 (10.61) mg/dL   MAE = 20.34 +/- 0.20 (6.95)   gRMSE = 37.10 +/- 0.33 (16.13)   Zone A = 74.96 +/- 0.57 (8.38)%   Zone A+B = 98.14 +/- 0.04 (2.00)%
120-min:  RMSE = 33.41 +/- 0.14 (12.13) mg/dL   MAE = 22.68 +/- 0.31 (7.94)   gRMSE = 40.81 +/- 0.34 (18.51)   Zone A = 70.80 +/- 0.77 (8.86)%   Zone A+B = 97.79 +/- 0.06 (2.48)%


In [9]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_resnet18/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_resnet18/gated_cgm_time_resnet18_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_resnet18/gated_cgm_time_resnet18_stability_summary.json
